# DAVID-Net Training â€” Kaggle

**Datasets:**
- **Training:** FakeAVCeleb (quadrant labels) + LAV-DF (localization labels)
- **Cross-dataset eval:** DFDC-10, DeepFakeTIMIT, Celeb-DF v2
- **Audio eval:** ASVspoof-2019, In-the-Wild, WaveFake

**Pipeline:**
1. Setup â†’ 2. Discover â†’ 3. Extract â†’ 4. Build manifests â†’ 5. Precompute features
6. QACP Stage 0 â†’ 7. Stage 1 (3 seeds) â†’ 8. Cross-dataset eval

**Architecture compliance:** VideoMAE + WavLM, cosine schedule, gradient checkpointing,
generator-balanced sampling, augmentation, focal loss, MI penalty.

In [ ]:
# Cell 1: GPU check + install deps
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU.")
!pip install -q transformers accelerate scikit-learn jiwer datasets

In [ ]:
# Cell 2: Clone repo
import os, sys
REPO = "/kaggle/working/david-net-av"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/MIHMahmudEli/david-net-av.git {REPO}
sys.path.insert(0, REPO)
print(f"Repo: {REPO}")

In [ ]:
# Cell 3: Load HF token
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF token loaded.")

In [ ]:
# Cell 4: Discover all 8 datasets
from pathlib import Path
KAGGLE_INPUT = Path("/kaggle/input")

datasets = {}
KNOWN = {
    "fakeavceleb": ["aicontentdetections/fakeavceleb-v1-2", "aicontentdetections/FakeAVCeleb_v1.2"],
    "lav-df": ["aicontentdetections/lav-df"],
    "dfdc-10": ["pranay22077/dfdc-10"],
    "deepfaketimit": ["fahimaislam1812/deepfaketimit", "fahimaislam1812/DeepfakeTIMIT"],
    "celeb-df-v2": ["reubensuju/celeb-df-v2"],
    "asvpoof-2019": ["anishsarkar22/asvpoof-2019-dataset-la"],
    "in-the-wild": ["abdallamohamed312/in-the-wild-audio-deepfake"],
    "wavefake": ["walimuhammadahmad/fakeaudio"],
}

for friendly, paths in KNOWN.items():
    for p in paths:
        for base in [KAGGLE_INPUT, KAGGLE_INPUT / "datasets"]:
            candidate = base / p
            if candidate.exists():
                datasets[friendly] = candidate
                print(f"  {friendly} -> {candidate}")
                break
        if friendly in datasets:
            break

print(f"\nFound {len(datasets)}/8 datasets.")

In [ ]:
# Cell 5: Extract compressed datasets
import zipfile

WORKING = Path("/kaggle/working")
DATA_DIR = WORKING / "data"
DATA_DIR.mkdir(exist_ok=True)

def extract_if_needed(name, src, dst):
    marker = dst / ".extracted"
    if marker.exists():
        print(f"  {name}: already done")
        return
    if any(src.rglob("*.mp4")) or any(src.rglob("*.wav")) or any(src.rglob("*.flac")) or any(src.rglob("*.avi")):
        print(f"  {name}: loose files, skipping")
        marker.touch()
        return
    for arch in src.rglob("*.zip"):
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with zipfile.ZipFile(arch) as zf:
                zf.extractall(dst)
            print("OK")
        except Exception as e:
            print(f"FAIL: {e}")
    marker.touch()

for name, path in datasets.items():
    dst = DATA_DIR / name
    dst.mkdir(exist_ok=True)
    extract_if_needed(name, path, dst)

print("Extraction done.")

In [ ]:
# Cell 6: Build ALL manifests
MANIFEST_DIR = WORKING / "manifests"
MANIFEST_DIR.mkdir(exist_ok=True)
SPLIT_DIR = WORKING / "splits"
SPLIT_DIR.mkdir(exist_ok=True)

# === FakeAVCeleb (existing converter) ===
fakeav_root = datasets.get("fakeavceleb")
if fakeav_root:
    for candidate in [fakeav_root, fakeav_root / "FakeAVCeleb_v1.2"]:
        if any((candidate / q).exists() for q in ["RealVideo-RealAudio", "FakeVideo-FakeAudio"]):
            fakeav_root = candidate
            break
    print(f"FakeAVCeleb: {fakeav_root}")
    !cd {REPO} && python scripts/build_manifest.py \
        --root {fakeav_root} \
        --out {MANIFEST_DIR}/fakeavceleb.jsonl \
        --splits-dir {SPLIT_DIR}/fakeavceleb --seed 42

# === All other datasets (unified converter) ===
CONVERTERS = [
    ("lav-df", "lav-df"),
    ("dfdc-10", "dfdc-10"),
    ("deepfaketimit", "deepfaketimit"),
    ("celeb-df-v2", "celeb-df-v2"),
    ("asvpoof-2019", "asvpoof-2019"),
    ("in-the-wild", "in-the-wild"),
    ("wavefake", "wavefake"),
]

for ds_name, dataset_key in CONVERTERS:
    root = datasets.get(dataset_key)
    if root and root.exists():
        print(f"\n--- {ds_name} ---")
        !cd {REPO} && python scripts/build_manifests.py \
            --dataset {dataset_key} \
            --root {root} \
            --out {MANIFEST_DIR}/{ds_name}.jsonl \
            --splits-dir {SPLIT_DIR}/{ds_name}
    else:
        print(f"  {ds_name}: NOT FOUND")

# === Summary ===
print("\n" + "="*50)
print("ALL MANIFESTS:")
for f in sorted(MANIFEST_DIR.glob("*.jsonl")):
    !wc -l {f}

In [ ]:
# Cell 7: Precompute SSL features for training sets (FakeAVCeleb + LAV-DF)
import yaml

FAKEAV_MANIFEST = str(MANIFEST_DIR / "fakeavceleb.jsonl")
FAKEAV_ROOT = str(datasets.get("fakeavceleb", ""))
LAVDF_MANIFEST = str(MANIFEST_DIR / "lav-df.jsonl")
LAVDF_ROOT = str(DATA_DIR / "lav-df")
FEAT_CACHE = WORKING / "feature_cache"
FEAT_CACHE.mkdir(exist_ok=True)

FEAT_CFG = {
    "video_backbone": "videomae", "audio_backbone": "wavlm",
    "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
    "freeze_blocks": 12, "freeze_feature_extractor": True,
    "d_model": 768, "n_frames": 16, "audio_len": 64000,
    "shard_root": None, "num_workers": 2,
}
feat_cfg_path = WORKING / "feat_cfg.yaml"
with open(feat_cfg_path, "w") as f:
    yaml.dump(FEAT_CFG, f)

# Extract features for both training datasets
TRAINING_DATASETS = [("FakeAVCeleb", FAKEAV_MANIFEST, FAKEAV_ROOT)]
lavdf_path = Path(LAVDF_MANIFEST)
if lavdf_path.exists():
    TRAINING_DATASETS.append(("LAV-DF", LAVDF_MANIFEST, LAVDF_ROOT))

for ds_name, manifest, root in TRAINING_DATASETS:
    n_cached = len(list(FEAT_CACHE.glob("*_vfeat.pt")))
    print(f"\nPrecomputing SSL features for {ds_name}...")
    !cd {REPO} && python -m src.data.extract_features \
        --config {feat_cfg_path} \
        --manifest {manifest} \
        --out {FEAT_CACHE} \
        --batch-size 4 \
        --root-dir {root}
    n_after = len(list(FEAT_CACHE.glob("*_vfeat.pt")))
    print(f"Cached features: {n_after} total clips (+{n_after - n_cached} from {ds_name})")

# Create combined training manifest (FakeAVCeleb + LAV-DF)
COMBINED_MANIFEST = WORKING / "train_combined.jsonl"
with open(COMBINED_MANIFEST, "w") as out:
    for manifest_path in [FAKEAV_MANIFEST, LAVDF_MANIFEST]:
        if Path(manifest_path).exists():
            with open(manifest_path) as f:
                for line in f:
                    out.write(line)
print(f"\nCombined training manifest: {COMBINED_MANIFEST}")
!wc -l {COMBINED_MANIFEST}

In [ ]:
# Cell 8: QACP Stage 0 config (NO feature_cache â€” needs raw data for transforms)
QACP_CONFIG = {
    "run_id": "qacp_stage0",
    "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
    "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
    "video_backbone": "videomae", "audio_backbone": "wavlm",
    "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
    "freeze_blocks": 12, "freeze_feature_extractor": True,
    "init_from": None,
    "n_frames": 16, "audio_len": 64000, "shard_root": None, "feature_cache": None,
    "train_manifest": str(COMBINED_MANIFEST),
    "root_dir": [FAKEAV_ROOT, LAVDF_ROOT],
    "modality_dropout": 0.0, "augment": False,
    "batch_size": 4, "num_workers": 2, "epochs": 20,
    "lr": 1e-4, "lr_encoder": 1e-5, "weight_decay": 1e-4,
    "warmup_epochs": 2, "gradient_checkpointing": True,
    "log_every": 10,
    "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
    "loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5, "sync": 0.3, "disentangle": 0.1},
    "grad_accum_steps": 8,
    "milestone_every": 5,
    "keep_milestones": 3,
    "qacp_temperature": 0.1, "seed": 42,
}

qacp_path = WORKING / "qacp_config.yaml"
with open(qacp_path, "w") as f:
    yaml.dump(QACP_CONFIG, f)
print(f"QACP: decodes from MP4 (raw data for Griffin-Lim/self-blending)")

In [ ]:
# Cell 9: Run QACP Stage 0
!cd {REPO} && python -m src.training.pretrain_qacp \
    --config {qacp_path} \
    --run-id {QACP_CONFIG['run_id']}

In [ ]:
# Cell 10: Stage 1 config (init from QACP best checkpoint on HF) -- 3 seeds
import glob
from pathlib import Path

# -- Safe defaults if earlier cells were skipped or simplified ----------------
try:
    _ = FEAT_CACHE
except NameError:
    FEAT_CACHE = WORKING / "feature_cache"
    FEAT_CACHE.mkdir(parents=True, exist_ok=True)
    print("FEAT_CACHE not set by earlier cell -- defaulting to", FEAT_CACHE)

try:
    _ = LAVDF_ROOT
except NameError:
    LAVDF_ROOT = str(WORKING / "data" / "lav-df")
    print("LAVDF_ROOT not set -- defaulting to", LAVDF_ROOT)

# Use feature_cache only if it actually has content
_feat_files = list(FEAT_CACHE.glob("*_vfeat.pt"))
USE_FEAT_CACHE = len(_feat_files) > 0
print(f"Feature cache: {len(_feat_files)} clips -- "
      f"{'ENABLED' if USE_FEAT_CACHE else 'DISABLED (raw decode)'}")

# -- Download the QACP best checkpoint from HF --------------------------------
from src.utils.hf_backup import HFBackup

QACP_RUN_ID = QACP_CONFIG["run_id"]   # "qacp_stage0"
_qacp_dl_dir = WORKING / "qacp_ckpt"
_qacp_dl_dir.mkdir(parents=True, exist_ok=True)

qacp_backup = HFBackup(run_id=QACP_RUN_ID, local_dir=str(WORKING))
print(f"Downloading QACP best checkpoint from HF (run_id={QACP_RUN_ID})...")

qacp_ckpt = qacp_backup.download_best(local_dir=str(_qacp_dl_dir))
if qacp_ckpt is None:
    print("  best.pt not found -- trying latest checkpoint...")
    qacp_ckpt = qacp_backup.download_latest(local_dir=str(_qacp_dl_dir))
if qacp_ckpt is None:
    print("WARNING: No QACP checkpoint found on HF.\n"
          "Stage 1 will train from RANDOM INIT (sub-optimal but safe).\n"
          "Re-run Cell 9 first to generate a QACP checkpoint.")
else:
    print(f"QACP checkpoint: {qacp_ckpt}")

# -- Build Stage-1 configs (3 seeds) ------------------------------------------
SEEDS = [42, 123, 456]
STAGE1_CONFIGS = []

for seed in SEEDS:
    cfg = {
        "run_id": f"stage1_seed{seed}",
        "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
        "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
        "video_backbone": "videomae", "audio_backbone": "wavlm",
        "video_model_name": "MCG-NJU/videomae-base",
        "audio_model_name": "microsoft/wavlm-base-plus",
        "freeze_blocks": 6, "freeze_feature_extractor": True,
        # Use downloaded QACP weights; None falls back to random init gracefully
        "init_from": qacp_ckpt if (qacp_ckpt and Path(qacp_ckpt).exists()) else None,
        "n_frames": 16, "audio_len": 64000, "shard_root": None,
        "feature_cache": str(FEAT_CACHE) if USE_FEAT_CACHE else None,
        "train_manifest": str(COMBINED_MANIFEST),
        "val_manifest": str(SPLIT_DIR / "fakeavceleb" / "val.jsonl"),
        "root_dir": FAKEAV_ROOT,
        "modality_dropout": 0.15, "augment": True,
        "batch_size": 4, "grad_accum_steps": 8, "num_workers": 2, "epochs": 30,
        "milestone_every": 10, "keep_milestones": 3,
        "lr": 1e-4, "lr_encoder": 1e-5, "weight_decay": 1e-4,
        "warmup_epochs": 2, "gradient_checkpointing": True,
        "log_every": 10,
        "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
        "loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5,
                         "sync": 0.3, "disentangle": 0.1},
        "seed": seed,
    }
    path = WORKING / f"stage1_seed{seed}_config.yaml"
    with open(path, "w") as f:
        yaml.dump(cfg, f)
    STAGE1_CONFIGS.append((seed, path, cfg))
    init_tag = "from QACP" if cfg["init_from"] else "random init"
    cache_tag = "feat_cache" if USE_FEAT_CACHE else "raw decode"
    print(f"Seed {seed}: {cfg['run_id']}  ({init_tag}, {cache_tag})")


In [ ]:
# Cell 11: Run Stage 1 training (3 seeds)
for seed, config_path, cfg in STAGE1_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Training seed {seed}...")
    print(f"{'='*60}")
    !cd {REPO} && python -m src.training.train \
        --config {config_path} \
        --run-id {cfg['run_id']}

In [ ]:
# Cell 12: Cross-dataset evaluation (ALL held-out datasets)
import json

# All eval datasets (NOT used in training)
EVAL_DATASETS = {
    # Cross-dataset generalization (Â§2 Table 2)
    "dfdc-10": MANIFEST_DIR / "dfdc-10.jsonl",
    "deepfaketimit": MANIFEST_DIR / "deepfaketimit.jsonl",
    "celeb-df-v2": MANIFEST_DIR / "celeb-df-v2.jsonl",
    # Audio-only evaluation (Â§3)
    "asvpoof-2019": MANIFEST_DIR / "asvpoof-2019.jsonl",
    "in-the-wild": MANIFEST_DIR / "in-the-wild.jsonl",
    "wavefake": MANIFEST_DIR / "wavefake.jsonl",
}

all_results = {}
for seed, config_path, stage1_cfg in STAGE1_CONFIGS:
    run_id = stage1_cfg["run_id"]
    best_ckpt = None
    for p in sorted(glob.glob(str(WORKING / f"runs/{run_id}_epoch*.pt")), reverse=True):
        best_ckpt = p
        break
    if not best_ckpt:
        print(f"Seed {seed}: no checkpoint found, skipping")
        continue

    seed_results = {}
    for ds_name, manifest in EVAL_DATASETS.items():
        if not manifest.exists():
            print(f"  {ds_name}: no manifest, skipping")
            continue
        print(f"\nSeed {seed} on {ds_name}...")
        report_path = WORKING / f"eval_{run_id}_{ds_name}.json"
        !cd {REPO} && python -m src.eval.evaluate \
            --config {config_path} \
            --checkpoint {best_ckpt} \
            --manifest {manifest} \
            --out {report_path}
        if report_path.exists():
            with open(report_path) as f:
                r = json.load(f)
            seed_results[ds_name] = {
                "video_auc": r["video"]["auc"],
                "audio_auc": r["audio"]["auc"],
                "quadrant_acc": r["quadrant"]["acc"],
            }
            print(f"  v_auc={r['video']['auc']:.4f} a_auc={r['audio']['auc']:.4f}")
    all_results[f"seed{seed}"] = seed_results

# === Summary ===
print("\n" + "="*70)
print("CROSS-DATASET EVALUATION SUMMARY")
print("="*70)
print(f"{'Dataset':<20} {'Type':<15} {'V-AUC':<10} {'A-AUC':<10} {'Quad-Acc':<10}")
print("-"*70)
for seed_key, ds_results in all_results.items():
    for ds, m in ds_results.items():
        dtype = "AV" if ds in ["dfdc-10", "deepfaketimit", "celeb-df-v2"] else "Audio"
        print(f"{ds:<20} {dtype:<15} {m['video_auc']:<10.4f} {m['audio_auc']:<10.4f} {m['quadrant_acc']:<10.4f}")

In [ ]:
# Cell 13: Verify HF backup
from huggingface_hub import HfApi
api = HfApi(token=hf_token)
for seed, _, cfg in STAGE1_CONFIGS:
    run = cfg["run_id"]
    try:
        files = list(api.list_repo_tree("MoshinAli/david-net-av-backup",
                                         path_in_repo=f"runs/{run}",
                                         repo_type="model", recursive=True))
        print(f"\n{run} on HF:")
        for f in files:
            if hasattr(f, 'path'): print(f"  {f.path}")
    except Exception as e:
        print(f"  {run}: {e}")